In [66]:
!pip install transformers datasets sacrebleu pandas torch

In [67]:
import torch
import pandas as pd

from datasets import load_dataset

from transformers import AutoTokenizer
from transformers import AutoModelForSeq2SeqLM

from sacrebleu.metrics import BLEU, CHRF

In [68]:
model_name = "Helsinki-NLP/opus-mt-en-mk"

print("Loading model...")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)

print("Device:", device)

Loading model...


/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Device: cpu


In [69]:
print("Loading OPUS-100 dataset...")

dataset = load_dataset(
    "Helsinki-NLP/opus-100",
    "en-mk",
    split="test[:50]"
)

Loading OPUS-100 dataset...


README.md: 0.00B [00:00, ?B/s]

en-mk/test-00000-of-00001.parquet:   0%|          | 0.00/196k [00:00<?, ?B/s]

en-mk/train-00000-of-00001.parquet:   0%|          | 0.00/76.4M [00:00<?, ?B/s]

en-mk/validation-00000-of-00001.parquet:   0%|          | 0.00/196k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/1000000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [70]:
en_sentences = []
mk_reference = []

for i in range(len(dataset)):

    item = dataset[i]["translation"]

    en_sentences.append(item["en"])
    mk_reference.append(item["mk"])

print("Examples loaded:", len(en_sentences))

Examples loaded: 50


In [71]:
def translate_text(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    ).to(device)

    with torch.no_grad():

        generated_tokens = model.generate(
            **inputs,
            max_length=512
        )

    translation = tokenizer.batch_decode(
        generated_tokens,
        skip_special_tokens=True
    )[0]

    return translation

In [72]:
our_translations = []

print("Translating...\n")

for i, sentence in enumerate(en_sentences):

    translated = translate_text(sentence)

    our_translations.append(translated)

    if (i + 1) % 10 == 0:
        print(f"{i+1}/{len(en_sentences)} done")

Translating...

10/50 done
20/50 done
30/50 done
40/50 done
50/50 done


In [76]:
df = pd.DataFrame({
    "English": en_sentences,
    "Reference MK": mk_reference,
    "Model Translation MK": our_translations
})

pd.set_option("display.max_colwidth", 150)
pd.set_option("display.width", 200)

print("=" * 50)
print("TRANSLATION RESULTS")
print("=" * 50)

print(df.head(10).to_string(index=False))

TRANSLATION RESULTS
                                                                                                                                                                                                                 English                                                                                                                                                                                                                                                         Reference MK                                                                                                                                                                                                                           Model Translation MK
                                                                                                                                                                                      Good, I wanna go with you on that.                                               

In [77]:
from sacrebleu.metrics import BLEU, CHRF

bleu = BLEU()

bleu_result = bleu.corpus_score(
    our_translations,
    [mk_reference]
)

chrf = CHRF()

chrf_result = chrf.corpus_score(
    our_translations,
    [mk_reference]
)

print("\n" + "=" * 40)
print("EVALUATION METRICS")
print("=" * 40)

print(f"BLEU score : {bleu_result}")
print(f"chrF score : {chrf_result}")


EVALUATION METRICS
BLEU score : BLEU = 39.02 63.8/42.5/32.7/26.2 (BP = 1.000 ratio = 1.014 hyp_len = 669 ref_len = 660)
chrF score : chrF2 = 63.08


In [78]:
df.to_csv(
    "mk_translation_results.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nResults saved to mk_translation_results.csv")


Results saved to mk_translation_results.csv
